# 07 — Đánh giá Retrieval và Câu trả lời (Evaluation)

Phase 7 đánh giá chất lượng của hệ thống Hue Foods RAG theo luồng trực quan:
```text
question -> retrieve -> build context -> generate -> judge -> report
```

- **Retrieval Evaluation**: Đánh giá khả năng tìm đúng tài liệu chứa từ khóa mong đợi (`MRR`, `nDCG`, `Keyword Coverage`).
- **Answer Evaluation**: Sử dụng `gpt-5.4-nano` để sinh câu trả lời và `gpt-5.4-mini` làm giám khảo (LLM-as-a-judge) chấm 3 tiêu chí: `accuracy`, `completeness`, `relevance` (thang điểm 1–5) cùng `feedback`.

## 1. Thiết lập đường dẫn backend

In [1]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Không tìm thấy thư mục backend/. Hãy mở notebook từ repo root hoặc thư mục notebooks/."
    )
print(f"backend on path: {sys.path[0]}")

backend on path: /home/minhhieu/hue_rag/backend


## 2. Import các hàm cần dùng

In [2]:
import asyncio
from evaluation.test import DEFAULT_TEST_FILE, load_tests
from evaluation.eval import (
    build_services,
    evaluate_answer,
    evaluate_retrieval,
    run_answer_batch,
    run_retrieval_batch,
)
from evaluation.evaluator import build_app

## 3. Đọc 20 câu hỏi thật từ test2.jsonl

In [3]:
questions = load_tests(DEFAULT_TEST_FILE)
print(f"Tổng số câu hỏi: {len(questions)}")

Tổng số câu hỏi: 20


## 4. Xem một câu hỏi mẫu

In [4]:
first_question = questions[0]
first_question

TestQuestion(question='Quán bún bò Mệ Kéo nằm ở đâu?', keywords=['Mệ Kéo', 'Bạch Đằng'], reference_answer='Quán bún bò Mệ Kéo nằm tại số 20 đường Bạch Đằng, phường Phú Cát, Thành phố Huế, khu vực chân cầu Gia Hội.', category='direct_fact')

## 5. Chạy retrieval cho một câu hỏi

In [5]:
services = build_services("dense_only")
retrieval_result = evaluate_retrieval(first_question, services)
retrieval_result

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'category': 'direct_fact',
 'question': 'Quán bún bò Mệ Kéo nằm ở đâu?',
 'keywords': 'Mệ Kéo | Bạch Đằng',
 'mrr': 1.0,
 'ndcg': 0.9074372343202424,
 'keywords_found': 2,
 'total_keywords': 2,
 'keyword_coverage': 100.0,
 'error': ''}

## 6. Ý nghĩa của MRR và nDCG

- **MRR (Mean Reciprocal Rank)**: Đánh giá vị trí xuất hiện đầu tiên của từ khóa trong danh sách kết quả ($1/\text{rank}$).
- **nDCG (Normalized Discounted Cumulative Gain)**: Đánh giá chất lượng xếp hạng tổng thể của các đoạn văn chứa từ khóa.
- **Keyword Coverage**: Tỷ lệ phần trăm từ khóa quan trọng được tìm thấy trong top kết quả.

## 7. Sinh và chấm một câu trả lời thật

Hệ thống truy xuất ngữ cảnh, sinh câu trả lời bằng `gpt-5.4-nano` và chấm điểm bằng `gpt-5.4-mini`. Từ khóa `await` cho phép chờ kết quả từ API trực tuyến mà không chặn các tiến trình khác.

In [6]:
answer_result = await evaluate_answer(first_question, services)
answer_result

{'category': 'direct_fact',
 'question': 'Quán bún bò Mệ Kéo nằm ở đâu?',
 'reference_answer': 'Quán bún bò Mệ Kéo nằm tại số 20 đường Bạch Đằng, phường Phú Cát, Thành phố Huế, khu vực chân cầu Gia Hội.',
 'generated_answer': 'Quán bún bò Mệ Kéo nằm ở **Số 20 đường Bạch Đằng, phường Phú Cát, Thành phố Huế** (ngay khu vực chân cầu Gia Hội).',
 'accuracy': 5,
 'completeness': 5,
 'relevance': 5,
 'feedback': 'Câu trả lời khớp hoàn toàn với đáp án tham khảo: đúng địa chỉ số 20 đường Bạch Đằng, phường Phú Cát, TP Huế và nêu đúng khu vực chân cầu Gia Hội. Trả lời trực tiếp, không có thông tin thừa.',
 'error': ''}

## 8. Xem ba điểm chất lượng và feedback

In [7]:
print(f"Accuracy: {answer_result['accuracy']}/5")
print(f"Completeness: {answer_result['completeness']}/5")
print(f"Relevance: {answer_result['relevance']}/5")
print(f"Feedback: {answer_result['feedback']}")

Accuracy: 5/5
Completeness: 5/5
Relevance: 5/5
Feedback: Câu trả lời khớp hoàn toàn với đáp án tham khảo: đúng địa chỉ số 20 đường Bạch Đằng, phường Phú Cát, TP Huế và nêu đúng khu vực chân cầu Gia Hội. Trả lời trực tiếp, không có thông tin thừa.


## 9. Chạy retrieval cho toàn bộ 20 câu hỏi

In [8]:
retrieval_rows, retrieval_summary = run_retrieval_batch(DEFAULT_TEST_FILE, 3)
retrieval_summary

{'questions': 20,
 'successful': 20,
 'failed': 0,
 'mrr': 0.7917,
 'ndcg': 0.802,
 'keyword_coverage': 96.67}

## 10. Chạy answer evaluation cho toàn bộ 20 câu hỏi

In [9]:
answer_rows, answer_summary = await run_answer_batch(DEFAULT_TEST_FILE, 3)
answer_summary

{'questions': 20,
 'successful': 19,
 'failed': 1,
 'accuracy': 4.53,
 'completeness': 4.21,
 'relevance': 4.42}

## 11. Giao diện Gradio

Để khởi chạy giao diện web đánh giá trực quan, gọi `build_app().launch(inbrowser=True)`.

In [ ]:
app = build_app()
app